<a href="https://colab.research.google.com/github/poojask27/MachineLearning-Lab2-73-/blob/main/foilalgo.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
import math

# ============================================================
# 1. READ DATASET
# ============================================================

df = pd.read_csv("pass_fail.csv")

print("=" * 60)
print("ORIGINAL DATASET")
print("=" * 60)

print(df.head())

print("\nColumns:")
print(df.columns.tolist())

print("\nDataset Shape:", df.shape)

print("\nMissing Values:")
print(df.isnull().sum())


# ============================================================
# 2. REMOVE STUDENT ID
# ============================================================

# student_id is only an identifier.
# It should not be used for classification.

df = df.drop(columns=["student_id"])


# ============================================================
# 3. CONVERT NUMERICAL VALUES INTO CATEGORIES
# ============================================================

# This makes the rules easier to understand.

def categorize(value, low, high):

    if value < low:
        return "Low"

    elif value < high:
        return "Medium"

    else:
        return "High"


df["attendance"] = df["attendance_pct"].apply(
    lambda x: categorize(x, 60, 80)
)

df["homework"] = df["homework_pct"].apply(
    lambda x: categorize(x, 60, 80)
)

df["midterm"] = df["midterm_score"].apply(
    lambda x: categorize(x, 60, 80)
)

df["study_hours"] = df["study_hours_per_week"].apply(
    lambda x: categorize(x, 5, 10)
)


# Keep only categorical attributes + target

df_rules = df[
    [
        "attendance",
        "homework",
        "midterm",
        "study_hours",
        "pass"
    ]
].copy()


print("\n" + "=" * 60)
print("PROCESSED DATASET")
print("=" * 60)

print(df_rules.head())


# ============================================================
# 4. CN2-STYLE RULE LEARNING
# ============================================================

def get_covered_data(data, condition):

    result = data.copy()

    for attribute, value in condition.items():

        result = result[
            result[attribute] == value
        ]

    return result


def cn2_rules(data):

    rules = []

    attributes = [
        column
        for column in data.columns
        if column != "pass"
    ]

    classes = data["pass"].unique()

    for class_value in classes:

        for attribute in attributes:

            for value in data[attribute].unique():

                condition = {
                    attribute: value
                }

                covered = get_covered_data(
                    data,
                    condition
                )

                if len(covered) == 0:
                    continue

                correct = sum(
                    covered["pass"] == class_value
                )

                accuracy = correct / len(covered)

                # Only keep reasonably strong rules
                if accuracy >= 0.70:

                    rules.append({
                        "condition": condition,
                        "class": class_value,
                        "accuracy": accuracy,
                        "coverage": len(covered)
                    })

    return rules


# Generate CN2 rules

cn2 = cn2_rules(df_rules)


# ============================================================
# 5. DISPLAY CN2 RULES
# ============================================================

print("\n" + "=" * 60)
print("CN2-STYLE CLASSIFICATION RULES")
print("=" * 60)

for i, rule in enumerate(cn2, 1):

    condition_text = " AND ".join(
        f"{attribute} = {value}"
        for attribute, value
        in rule["condition"].items()
    )

    class_name = (
        "PASS"
        if rule["class"] == 1
        else "FAIL"
    )

    print(
        f"\nRule {i}: "
        f"IF {condition_text} "
        f"THEN Result = {class_name}"
    )

    print(
        f"Accuracy : {rule['accuracy']:.2f}"
    )

    print(
        f"Coverage : {rule['coverage']}"
    )


# ============================================================
# 6. FOIL INFORMATION GAIN
# ============================================================

def foil_gain(
    positive_before,
    negative_before,
    positive_after,
    negative_after
):

    if positive_after == 0:
        return -1

    if positive_before == 0:
        return -1

    if positive_before + negative_before == 0:
        return -1

    if positive_after + negative_after == 0:
        return -1

    old_probability = (
        positive_before /
        (positive_before + negative_before)
    )

    new_probability = (
        positive_after /
        (positive_after + negative_after)
    )

    if old_probability == 0 or new_probability == 0:
        return -1

    gain = positive_after * (
        math.log2(new_probability)
        - math.log2(old_probability)
    )

    return gain


# ============================================================
# 7. FOIL ALGORITHM
# ============================================================

def foil(data):

    attributes = [
        column
        for column in data.columns
        if column != "pass"
    ]

    # pass = 1 is positive
    positive_examples = data[
        data["pass"] == 1
    ].copy()

    negative_examples = data[
        data["pass"] == 0
    ].copy()

    learned_rules = []

    while len(positive_examples) > 0:

        rule = {}

        current_positive = positive_examples.copy()
        current_negative = negative_examples.copy()

        # ----------------------------------------------------
        # Find conditions until negative examples are removed
        # ----------------------------------------------------

        while len(current_negative) > 0:

            best_gain = -1
            best_attribute = None
            best_value = None

            for attribute in attributes:

                if attribute in rule:
                    continue

                for value in data[attribute].unique():

                    positive_after = current_positive[
                        current_positive[attribute] == value
                    ]

                    negative_after = current_negative[
                        current_negative[attribute] == value
                    ]

                    gain = foil_gain(
                        len(current_positive),
                        len(current_negative),
                        len(positive_after),
                        len(negative_after)
                    )

                    if gain > best_gain:

                        best_gain = gain
                        best_attribute = attribute
                        best_value = value

            # No useful condition
            if best_attribute is None:
                break

            # Add selected condition
            rule[best_attribute] = best_value

            current_positive = current_positive[
                current_positive[best_attribute]
                == best_value
            ]

            current_negative = current_negative[
                current_negative[best_attribute]
                == best_value
            ]

        # ----------------------------------------------------
        # Store rule
        # ----------------------------------------------------

        if len(current_positive) > 0:

            learned_rules.append({
                "condition": rule.copy(),
                "class": 1
            })

            # Remove positive examples covered by rule

            mask = pd.Series(
                True,
                index=positive_examples.index
            )

            for attribute, value in rule.items():

                mask &= (
                    positive_examples[attribute]
                    == value
                )

            positive_examples = positive_examples[
                ~mask
            ]

        else:
            break

    return learned_rules


# ============================================================
# 8. RUN FOIL
# ============================================================

foil_rules = foil(df_rules)


# ============================================================
# 9. DISPLAY FOIL RULES
# ============================================================

print("\n" + "=" * 60)
print("FOIL FIRST-ORDER RULES")
print("=" * 60)

for i, rule in enumerate(foil_rules, 1):

    predicates = []

    for attribute, value in rule["condition"].items():

        predicate = (
            f"{attribute}(Student, {value})"
        )

        predicates.append(predicate)

    body = ", ".join(predicates)

    print(
        f"\nRule {i}:"
    )

    print(
        f"pass(Student) :- {body}."
    )


# ============================================================
# 10. FOIL PREDICTION
# ============================================================

def predict_foil(example, rules):

    for rule in rules:

        matches = True

        for attribute, value in rule["condition"].items():

            if example[attribute] != value:

                matches = False
                break

        if matches:

            return 1

    return 0


# ============================================================
# 11. TEST WITH A NEW STUDENT
# ============================================================

new_student = {
    "attendance": "High",
    "homework": "High",
    "midterm": "High",
    "study_hours": "High"
}


prediction = predict_foil(
    new_student,
    foil_rules
)


print("\n" + "=" * 60)
print("NEW STUDENT PREDICTION")
print("=" * 60)

print("\nStudent:")
print(new_student)

if prediction == 1:
    print("\nPredicted Result: PASS")
else:
    print("\nPredicted Result: FAIL")


# ============================================================
# 12. SUMMARY
# ============================================================

print("\n" + "=" * 60)
print("SUMMARY")
print("=" * 60)

print("Number of students :", len(df_rules))
print("Number of features :", len(df_rules.columns) - 1)
print("Target variable    : pass")
print("CN2 rules generated:", len(cn2))
print("FOIL rules generated:", len(foil_rules))

ORIGINAL DATASET
   student_id  attendance_pct  homework_pct  midterm_score  \
0           1              95            92             88   
1           2              88            85             79   
2           3              60            55             58   
3           4              72            70             65   
4           5              40            45             50   

   study_hours_per_week  pass  
0                    12     1  
1                    10     1  
2                     4     0  
3                     6     1  
4                     3     0  

Columns:
['student_id', 'attendance_pct', 'homework_pct', 'midterm_score', 'study_hours_per_week', 'pass']

Dataset Shape: (100, 6)

Missing Values:
student_id              0
attendance_pct          0
homework_pct            0
midterm_score           0
study_hours_per_week    0
pass                    0
dtype: int64

PROCESSED DATASET
  attendance homework midterm study_hours  pass
0       High     High    High   